# Week 23, Day 4: Chaos from a simple rule: the logistic map

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w023/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The period of the attractor

Write `period(r)`: iterate from x = 0.5 for 5,000 steps to settle on the attractor, then iterate up to 64 more times and return the first p at which x returns within 1e-8 of where it was, or None.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def period(r):
    x = 0.5
    for _ in range(5000):
        x = r * x * (1 - x)
    x0 = x
    for p in range(1, 65):
        x = ...
        if ...:
            return p
    return None

for r in (2.8, 3.2, 3.5, 3.55, 3.83, 3.9):
    print(f"r = {r}: period {period(r)}")

In [ ]:
import numpy as np
def test_cascade():
    "periods are 1, 2, 4, 8 at r = 2.8, 3.2, 3.5, 3.55, and 3 at r = 3.83"
    assert [period(r) for r in (2.8, 3.2, 3.5, 3.55, 3.83)] == [1, 2, 4, 8, 3]

def test_chaos():
    "at r = 3.9 no period up to 64 is found"
    assert period(3.9) is None

test_cascade()
test_chaos()
print("All checks passed")

### Exercise: Computing the universal ratio

With g(r, n) = f_r applied 2ⁿ times to 0.5, minus 0.5, find R[n] for n = 2 to 6 by Newton's method (50 steps, derivative by central difference with h = 1e-9), starting from the guess R[n−1] + (R[n−1] − R[n−2]) / 4.7. Then compute `ratios`, the successive ratios (R[n] − R[n−1]) / (R[n+1] − R[n]).

In [ ]:
def g(r, n):
    x = 0.5
    for _ in range(2 ** n):
        x = r * x * (1 - x)
    return x - 0.5

R = [2.0, 1 + np.sqrt(5)]
for n in range(2, 7):
    r = R[-1] + (R[-1] - R[-2]) / 4.7
    for _ in range(50):
        h = 1e-9
        r = ...
    R.append(r)
R = np.array(R)
ratios = ...
print("superstable r:", R.round(6)); print("ratios:", ratios.round(4))

In [ ]:
import numpy as np
def test_R():
    "R[1] = 1 + sqrt(5) and R[2] is about 3.49856"
    assert abs(R[1] - (1 + np.sqrt(5))) < 1e-9 and abs(R[2] - 3.49856) < 1e-5

def test_delta():
    "the last ratio is within 0.01 of Feigenbaum's 4.6692"
    assert abs(ratios[-1] - 4.6692) < 0.01

test_R()
test_delta()
print("All checks passed")

### Exercise: The Lyapunov exponent

Write `lyapunov(r)`: from x = 0.3, iterate 20,000 times and average ln|r(1 − 2x)| over all but the first 1,000 steps. Then iterate two orbits at r = 4 from 0.3 and 0.3 + 1e-10 and store in `n_split` the first step at which they differ by more than 0.1.

In [ ]:
def lyapunov(r):
    x, s = 0.3, 0.0
    for k in range(20000):
        x = r * x * (1 - x)
        if k >= 1000:
            s += ...
    return s / 19000

a, b, n_split = 0.3, 0.3 + 1e-10, None
for n in range(1, 200):
    a, b = 4 * a * (1 - a), 4 * b * (1 - b)
    if ...:
        n_split = n; break
print({r: round(float(lyapunov(r)), 4) for r in (3.2, 3.7, 3.83, 4.0)}, f";  ln 2 = {np.log(2):.4f};  orbits split after {n_split} steps")
rs = np.linspace(2.8, 4, 80); plt.plot(rs, [lyapunov(r) for r in rs]); plt.axhline(0, c="k", lw=0.8)
plt.xlabel("r"); plt.ylabel("Lyapunov exponent"); plt.show()

In [ ]:
import numpy as np
def test_values():
    "lambda(4) is within 0.01 of ln 2; lambda is negative at 3.2 and 3.83 and positive at 3.7"
    assert abs(lyapunov(4.0) - np.log(2)) < 0.01 and lyapunov(3.2) < 0 and lyapunov(3.83) < 0 and lyapunov(3.7) > 0

def test_split():
    "two orbits 1e-10 apart at r = 4 first differ by more than 0.1 after 25 to 45 steps"
    assert 25 <= n_split <= 45

test_values()
test_split()
print("All checks passed")